# pyclesperanto interoperability with CuPy and PyTorch

This notebook shows how to move array data between `pyclesperanto`, `cupy`, and `torch`,
in both directions, using two mechanisms:

1. The **NumPy API** (`.get()` / `cle.push()`), which always goes through a host (CPU) copy.
2. The **DLPack protocol** (`__dlpack__` / `from_dlpack`), which can share the device memory
   directly (zero-copy) when both libraries run on the same GPU backend (CUDA), and otherwise
   falls back to a copy.

No processing/operations are demonstrated here, only data transfer.

In [ ]:
import numpy as np
import pyclesperanto as cle

cle.select_device()

## 1. pyclesperanto ↔ CuPy

### Via the NumPy API (host copy)

`cle.push()` accepts anything array-like, including CuPy arrays, by first converting it to NumPy
on the host. `.get()` (or `np.asarray()`) brings a `cle.Array` back to a NumPy array, which can then
be handed to CuPy.

In [ ]:
import cupy as cp

cp_array = cp.arange(12, dtype=cp.float32).reshape(3, 4)
cle_array = cle.push(cp_array)     # CuPy -> pyclesperanto (via host copy)
cle_array

In [ ]:
back_to_cp = cp.asarray(cle_array.get())   # pyclesperanto -> CuPy (via host copy)
back_to_cp

### Via DLPack (zero-copy)

If `pyclesperanto` is used with CUDA backend, `from_dlpack` can shares the underlying device pointer directly, without copying. 
If backends differ, it will falls back to a copy.

In [ ]:
cle_from_dlpack = cle.Array.from_dlpack(cp_array)   # CuPy -> pyclesperanto (DLPack)
cle_from_dlpack

In [ ]:
cp_from_dlpack = cp.from_dlpack(cle_from_dlpack)    # pyclesperanto -> CuPy (DLPack)
cp_from_dlpack

## 2. pyclesperanto ↔ PyTorch

### Via the NumPy API (host copy)

In [ ]:
import torch

torch_device = "cuda" if torch.cuda.is_available() else "cpu"
torch_tensor = torch.arange(12, dtype=torch.float32, device=torch_device).reshape(3, 4)

cle_array = cle.push(torch_tensor)   # torch -> pyclesperanto (via host copy)
cle_array

In [ ]:
back_to_torch = torch.tensor(cle_array.get())   # pyclesperanto -> torch (via host copy)
back_to_torch

### Via DLPack (potentially zero-copy)

If `pyclesperanto` and `pytorch` share the same backend: CUDA or Metal (mps), `from_dlpack` can shares the underlying device pointer directly, without copying. 
If backends differ, it will falls back to a copy.

In [ ]:
cle_from_dlpack = cle.Array.from_dlpack(torch_tensor)   # torch -> pyclesperanto (DLPack)
cle_from_dlpack

In [ ]:
torch_from_dlpack = torch.from_dlpack(cle_from_dlpack)  # pyclesperanto -> torch (DLPack)
torch_from_dlpack

## Summary

| Direction | NumPy API | DLPack |
|---|---|---|
| CuPy → pyclesperanto | `cle.push(cp_array)` | `cle.Array.from_dlpack(cp_array)` |
| pyclesperanto → CuPy | `cp.asarray(cle_array.get())` | `cp.from_dlpack(cle_array)` |
| torch → pyclesperanto | `cle.push(torch_tensor)` | `cle.Array.from_dlpack(torch_tensor)` |
| pyclesperanto → torch | `torch.tensor(cle_array.get())` | `torch.from_dlpack(cle_array)` |

The NumPy-API route always round-trips through host memory. The DLPack route is zero-copy when
both libraries share the same CUDA device, and falls back to a copy otherwise.